# 04 Statistical Analysis

Apply rigorous statistical methods to the cleaned **Retail Store Sales** dataset.

**Analyses in this notebook:**
1. Welch's t-test — Total Spent: Discount Applied vs. No Discount
2. Chi-squared test — Independence of Category and Payment Method
3. Chi-squared test — Independence of Location and Discount Applied
4. Point-biserial correlation — numeric features vs. discount_applied
5. Discount Impact Index by Category (observed vs baseline discount rates)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['transaction_date'])
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
discounted = df[df['discount_applied'] == 1]
no_discount = df[df['discount_applied'] == 0]
print(f'Discounted transactions : {len(discounted):,}')
print(f'No-discount transactions: {len(no_discount):,}')
print(f'Discount rate           : {len(discounted)/len(df)*100:.2f}%')

## 4.1 Welch's T-Test — Total Spent (Discount vs No Discount)

In [ ]:
t_stat, p_value = stats.ttest_ind(
    discounted['total_spent'],
    no_discount['total_spent'],
    equal_var=False  # Welch's t-test — does not assume equal variance
)

print('=== Welch\'s T-Test: Total Spent (Discounted vs. No Discount) ===')
print(f'  Discounted mean : ${discounted["total_spent"].mean():,.2f}')
print(f'  No Discount mean: ${no_discount["total_spent"].mean():,.2f}')
print(f'  t-statistic     : {t_stat:.4f}')
print(f'  p-value         : {p_value:.6f}')
significance = 'SIGNIFICANT (reject H₀)' if p_value < 0.05 else 'NOT significant (fail to reject H₀)'
print(f'  Result          : {significance} at α=0.05')
print()
print('Business interpretation:')
print('  Determines whether offering a discount results in a statistically')
print('  significant difference in the average transaction value.')

## 4.2 Chi-Squared Test — Independence of Category and Payment Method

In [ ]:
contingency_cat_pay = pd.crosstab(df['category'], df['payment_method'])
chi2_cp, p_cp, dof_cp, _ = stats.chi2_contingency(contingency_cat_pay)

print('=== Chi-Squared Test: Category vs Payment Method ===')
print(f'  Chi²  : {chi2_cp:.4f}')
print(f'  DoF   : {dof_cp}')
print(f'  p     : {p_cp:.6f}')
result_cp = 'DEPENDENT — payment preferences differ by category' if p_cp < 0.05 else 'INDEPENDENT'
print(f'  Result: {result_cp}')
print()
print('Contingency table:')
print(contingency_cat_pay)

## 4.3 Chi-Squared Test — Independence of Location and Discount Applied

In [ ]:
contingency_loc_disc = pd.crosstab(df['location'], df['discount_applied'])
chi2_ld, p_ld, dof_ld, _ = stats.chi2_contingency(contingency_loc_disc)

print('=== Chi-Squared Test: Location vs Discount Applied ===')
print(f'  Chi²  : {chi2_ld:.4f}')
print(f'  DoF   : {dof_ld}')
print(f'  p     : {p_ld:.6f}')
result_ld = 'DEPENDENT — discounts are not distributed equally across Online/In-store' if p_ld < 0.05 else 'INDEPENDENT'
print(f'  Result: {result_ld}')
print()
print('Contingency table:')
print(contingency_loc_disc)

## 4.4 Point-Biserial Correlation — Numeric Features vs. Discount Applied

In [ ]:
numeric_features = ['price_per_unit', 'quantity', 'total_spent', 'avg_unit_price']

results = []
for feat in numeric_features:
    if feat not in df.columns:
        continue
    corr_val, p_val = stats.pointbiserialr(df['discount_applied'], df[feat])
    results.append({
        'feature': feat,
        'corr': round(corr_val, 4),
        'p_value': round(p_val, 6),
        'significant': 'Yes' if p_val < 0.05 else 'No'
    })

corr_df = pd.DataFrame(results).sort_values('corr', key=abs, ascending=False)
print('Point-Biserial Correlation with discount_applied:')
print(corr_df.to_string(index=False))

## 4.5 Discount Impact Index by Category

Risk Index = (Observed discount rate in category) / (Overall discount rate)

In [ ]:
overall_discount_rate = df['discount_applied'].mean()

impact_index = (
    df.groupby('category')['discount_applied']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'discounted_txns', 'count': 'total_txns'})
    .assign(
        expected_discounts=lambda x: x['total_txns'] * overall_discount_rate,
        category_discount_rate=lambda x: x['discounted_txns'] / x['total_txns'] * 100,
        impact_index=lambda x: x['discounted_txns'] / x['expected_discounts']
    )
    .sort_values('impact_index', ascending=False)
    .reset_index()
)

print(f'Overall discount rate (baseline): {overall_discount_rate * 100:.2f}%')
print()
print('Discount Impact Index by Category:')
print(impact_index[['category','discounted_txns','total_txns','category_discount_rate','impact_index']].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#e05c5c' if r > 1.0 else '#4e9af1' for r in impact_index['impact_index']]
ax.bar(impact_index['category'], impact_index['impact_index'], color=colors, edgecolor='white')
ax.axhline(1.0, linestyle='--', color='black', linewidth=1.2, label='Baseline (index = 1)')
ax.set_title('Discount Impact Index by Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Impact Index (>1 = above average discount rate)')
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Statistical Analysis Summary

| Test | Variable | Result | Business Meaning |
|---|---|---|---|
| Welch's t-test | Total Spent | *see output* | Does a discount meaningfully change order size? |
| Chi-squared | Category vs Payment | *see output* | Payment preferences are linked to products |
| Chi-squared | Location vs Discount | *see output* | Channel distribution of discounts is not uniform |
| Point-Biserial | Numeric Features | *see output* | Identifies numeric correlations with discount usage |
| Impact Index | Category | *see chart* | Identifies categories that rely heavily on discounts |

**Next step →** `05_final_load_prep.ipynb`